# Inference of Taxonomic and Functional Composition of Microbial Communities

We will:

1. Download dataset (using web scrapping)
2. Create Manifest file for Qiime2
3. Install dependencies and Qiime2
4. Run Qiime2 analysis


# Clone Github Repo

We start by cloning our repo to be able to store our data orderly

In [ ]:
# 1. Clone GitHub repo to Colab
REPO_URL = "https://github.com/hector-ahg/predictive-microbiology-course.git"

!git clone $REPO_URL
%cd predictive-microbiology-course

# Load packages

In [ ]:
import requests #The requests module allows you to send HTTP requests using Python
from bs4 import BeautifulSoup #Beautiful Soup is a Python library for pulling data out of HTML and XML files. It provides tools for parsing and navigating the parse tree, making it easy to extract data from web pages.
from urllib.parse import urljoin, urlparse #The urllib.parse module provides functions for manipulating URLs, including joining base URLs with relative URLs to create absolute URLs.
import os #The os module provides a way of using operating system dependent functionality like reading or writing to the file system, handling file paths, and managing directories.
from pathlib import Path #The pathlib allows you to work with file paths in an object-oriented way, making it easier to manipulate and interact with file system paths across different operating systems.
import subprocess
import sys
import numpy as np
import pandas as pd

# Folder structure

In [ ]:
# Base directories 
BASE_DIR = Path.cwd().resolve() #project_root

print("Current working directory:", BASE_DIR)

DATA_DIR = BASE_DIR / "dataset" 
DATA_IBDMDB = DATA_DIR /"ibdmdb_hmp2"
PATH_sequences = DATA_IBDMDB / "raw_sequences"
PATH_manifest =  DATA_IBDMDB / "manifests"
PATH_results = BASE_DIR / "results"
QC_DIR = PATH_results / "quality_control" # demux files 

# Create folders if they does not exist
PATH_manifest.mkdir(parents=True, exist_ok=True)
PATH_sequences.mkdir(parents=True, exist_ok=True)
PATH_results.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)


# Downloading IBDMDB Data

This notebook demonstrates how to download raw sequencing data from the Inflammatory Bowel Disease Multi-omics Database (https://ibdmdb.org/) using basic web scraping techniques in Python.

## Goals

1. Become familiar with sending HTTP requests using 'requests'.
2. Learn how to extract data from HTML using 'BeautifulSoup'.

In [ ]:
# Download raw files (sequences) from IBDMDB

url= "https://ibdmdb.org/downloads/html/rawfiles_16s_2018-01-08.html"

response = requests.get(url)
print('Status code:', response.status_code) # if 200, the request was successful

In [ ]:
# Parse the HTML content of the page to find the download links for the raw files

soup = BeautifulSoup(response.text, "html.parser")
print(soup) # print the HTML content of the page to understand its structure and identify the relevant elements containing the download links.

In [ ]:

links = []
# iterate over the elements of the webpage 'a' finds all anchor tags with an href attribute, which typically represent hyperlinks in HTML. The href attribute contains the URL of the linked resource.
for a in soup.find_all('a', href=True): 
    link = a['href'] # extract the URL of the linked resource
    # check if the link ends with '.fastq.gz', which is a common file extension for compressed FASTQ files (sequences), often used for storing biological sequence data.
    if link.endswith('.fastq.gz'):
        full_link = urljoin(url, link) # Convert relative URL to absolute URL
        links.append(full_link)
        print('Download link found:', full_link)

In [ ]:
# print the current working directory to confirm where the files will be saved
#print(os.getcwd()) 

# Change the current working directory to the desired output directory
#os.chdir(PATH_sequences)

#print(os.getcwd())

In [ ]:
# Download each file from the list of links

#n= len(links) # get the total number of download links found on the webpage
n = 10  # number of files to download (uncomment if you want to download a subset of size n)

for i, url in enumerate(links[:n]): # iterate over the first n links in the list of download links
    response = requests.get(url, stream=True) # send a GET request to the URL to download the file. The stream=True parameter allows you to download the file in chunks, which is useful for large files.
    response.raise_for_status() # check if the request was successful (status code 200). If the request was not successful, this will raise an HTTPError exception.

    filename = os.path.basename(urlparse(url).path) # extract the filename from the URL using urlparse to parse the URL and os.path.basename to get the base name of the file from the path component of the URL.
    filepath = PATH_sequences/filename # create the full file path by joining the current working directory with the filename.

    print(f"Downloading file {i}: {filename}") # print a message indicating which file is being downloaded, including the index of the file in the list and the filename.

    with open(filepath, "wb") as f: # open the file in binary write mode to save the downloaded content. The with statement ensures that the file is properly closed after writing.
        for chunk in response.iter_content(chunk_size=8192): # iterate over the content of the response in chunks of 8192 bytes (8 KB) using the iter_content method. This allows you to download large files without consuming too much memory.
            f.write(chunk) # write each chunk of data to the file until the entire file is downloaded.

    print("File downloaded successfully:", filename)
print("All files downloaded successfully.")

# Create manifest

In [ ]:
# Change working directory
os.chdir(PATH_sequences)

print("Now working in:", os.getcwd())

# Collect files
fastq_files = list(PATH_sequences.glob("*.fastq.gz"))

In [ ]:
# Qiime2 requires a manifest of sequences to work
# Build manifest
manifest = pd.DataFrame({
    "sample-id": [f.stem.replace(".fastq", "") for f in fastq_files],
    "absolute-filepath": [str(f.resolve()) for f in fastq_files]
})

# Optional: sort for reproducibility
manifest = manifest.sort_values("sample-id")

# Save manifest file in PATH_manifest
manifest.to_csv(PATH_manifest/ "manifest_in_colab.tsv", sep="\t", index=False)

manifest.head()


In [ ]:
# Check manifest file exists
manifest_file = PATH_manifest / "manifest_in_colab.tsv"

if not manifest_file.is_file():
    raise FileNotFoundError("Manifest file not found!")

print("Manifest file found:", manifest_file)
# print number of samples
len(manifest)

# Setup 

Installation of Miniconda, Mamba, QIIME2, and their corresponding helper functions.

## Install Miniconda

In [ ]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /opt/conda

PREFIX=/opt/conda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda


In [ ]:
os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

In [ ]:
!conda --version

conda 26.3.2


In [ ]:
# Accept Terms of Service
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


## Install Mamba


In [ ]:
%%capture # to silece warning messages
!conda install -y -n base -c conda-forge mamba

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: / - \ | / - \ done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - mamba


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    conda-26.3.2               |  py313h78bf25f_1         1.2 MB  conda-forge
    cpp-expected-1.3.1         |       h171cf75_0          24 KB  conda-forge
    libarchive-3.8.7           |       hb3cce40_0         867 KB
    libcurl-8.20.0             |       hd8fa685_1         491 KB
    libmamba-2.6.0             |       hd28c85e_0         2.7 MB  conda-forge
    libmamba-spdlog-2.6.0      |       hf859cbd_0          20 KB  conda-forge
    libmambapy-2.6.0           |  py3

In [ ]:
!mamba --version

2.6.0


## Install QIMME2

In [ ]:
# This command can take up to 40 minutes to complete.
# IMPORTANT: Around 30 minutes into the execution, you will be prompted to type 'Y' to continue the installation.
!mamba env create -n qiime2-amplicon-2024.10 --file https://data.qiime2.org/distro/amplicon/qiime2-amplicon-2024.10-py310-linux-conda.yml

Streaming output truncated to the last 5000 lines.
Extracting  (682)  ⣾  [+] 1m:0.4s
Extracting  (682)  ⣾  [+] 1m:0.5s
Extracting  (680)  ⣾  [+] 1m:0.6s
Extracting  (680)  ⣾  [+] 1m:0.7s
Extracting  (680)  ⣾  [+] 1m:0.8s
Extracting  (679)  ⣾  [+] 1m:0.9s
Extracting  (678)  ⣾  [+] 1m:1.0s
Extracting  (678)  ⣾  [+] 1m:1.1s
Extracting  (678)  ⣾  [+] 1m:1.2s
Extracting  (677)  ⣾  [+] 1m:1.3s
Extracting  (676)  ⣾  [+] 1m:1.4s
Extracting  (676)  ⣾  [+] 1m:1.5s
Extracting  (675)  ⣾  [+] 1m:1.6s
Extracting  (674)  ⣾  [+] 1m:1.7s
Extracting  (674)  ⣾  [+] 1m:1.8s
Extracting  (674)  ⣾  [+] 1m:1.9s
Extracting  (674)  ⣾  [+] 1m:2.0s
Extracting  (673)  ⣾  [+] 1m:2.1s
Extracting  (673)  ⣾  [+] 1m:2.2s
Extracting  (673)  ⣾  [+] 1m:2.3s
Extracting  (673)  ⣾  [+] 1m:2.4s
Extracting  (672)  ⣾  [+] 1m:2.5s
Extracting  (671)  ⣾  [+] 1m:2.6s
Extracting  (671)  ⣾  [+] 1m:2.7s
Extracting  (671)  ⣾  [+] 1m:2.8s
Extracting  (671)  ⣾  [+] 1m:2.9s
Extracting  (671)  ⣾  [+] 1m:3.0s
Extracting  (671)  ⣾  [+] 1m:3.

In [ ]:
# Check installation
!source /opt/conda/bin/activate qiime2-amplicon-2024.10 && qiime info

QIIME is caching your current deployment for improved performance. This may take a few moments and should only happen once per deployment.
System versions
Python version: 3.10.14
QIIME 2 release: 2024.10
QIIME 2 version: 2024.10.1
q2cli version: 2024.10.1

Installed plugins
alignment: 2024.10.0
composition: 2024.10.0
cutadapt: 2024.10.0
dada2: 2024.10.0
deblur: 2024.10.0
demux: 2024.10.0
diversity: 2024.10.0
diversity-lib: 2024.10.0
emperor: 2024.10.0
feature-classifier: 2024.10.0
feature-table: 2024.10.0
fragment-insertion: 2024.10.0
longitudinal: 2024.10.0
metadata: 2024.10.0
phylogeny: 2024.10.0
quality-control: 2024.10.0
quality-filter: 2024.10.0
rescript: 2024.10.0
sample-classifier: 2024.10.0
stats: 0+unknown
taxa: 2024.10.0
types: 2024.10.0
vizard: 0.0.1.dev0
vsearch: 2024.10.0

Application config directory
/opt/conda/envs/qiime2-amplicon-2024.10/var/q2cli

Config
Config Source: /opt/conda/envs/qiime2-amplicon-2024.10/etc/qiime2_config.toml

Getting help
To get help with QIIME 2

## Helper function for QIIME2

In [ ]:
# Helper function for running QIIME2 (it does not interfer with PIICRUST2)
def qiime(cmd):
    full_cmd = f"source /opt/conda/bin/activate qiime2-amplicon-2024.10 && {cmd}"
    result = subprocess.run(
        full_cmd,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

## Is QIIME2 working?

In [ ]:
qiime("qiime --version")

q2cli version 2024.10.1
Run `qiime info` for more version details.



0

In [ ]:
qiime("qiime tools --help")

Usage: qiime tools [OPTIONS] COMMAND [ARGS]...

  Tools for working with QIIME 2 files.

Options:
  --help      Show this message and exit.

Commands:
  cache-create              Create an empty cache at the given location.
  cache-fetch               Fetches an artifact out of a cache into a .qza.
  cache-garbage-collection  Runs garbage collection on the cache at the
                            specified location.
  cache-import              Imports data into an Artifact in the cache under a
                            key.
  cache-remove              Removes a given key from a cache.
  cache-status              Checks the status of the cache.
  cache-store               Stores a .qza in the cache under a key.
  cast-metadata             Designate metadata column types.
  citations                 Print citations for a QIIME 2 result.
  export                    Export data from a QIIME 2 Artifact or a
                            Visualization
  extract                   Extract a QI

0

# QIIME2 analysis

In [ ]:
qiime(f"""
qiime tools import \
  --type 'SampleData[SequencesWithQuality]' \
  --input-path manifest.tsv \
  --output-path {QC_DIR}/demux.qza \
  --input-format SingleEndFastqManifestPhred33V2
""")


Usage: qiime tools import [OPTIONS]

  Import data to create a new QIIME 2 Artifact. See https://docs.qiime2.org/
  for usage examples and details on the file types and associated semantic
  types that can be imported.

Options:
  --type TEXT             The semantic type of the artifact that will be
                          created upon importing. Use `qiime tools list-types`
                          to see what importable semantic types are available
                          in the current deployment.                [required]
  --input-path PATH       Path to file or directory that should be imported.
                                                                    [required]
  --output-path ARTIFACT  Path where output artifact should be written.
                                                                    [required]
  --input-format TEXT     The format of the data to be imported. If not
                          provided, data must be in the format expected by the
   

1

In [ ]:
# Quality summary
qiime(f"""
qiime demux summarize \
  --i-data {QC_DIR}/demux.qza \
  --o-visualization {QC_DIR}/demux-summary.qzv
""")
print("Check visualization of sequence quality: https://view.qiime2.org/")



Usage: qiime demux summarize [OPTIONS]

  Summarize counts per sample for all samples, and generate interactive
  positional quality plots based on `n` randomly selected sequences.

Inputs:
  --i-data ARTIFACT SampleData[SequencesWithQuality |
    PairedEndSequencesWithQuality | JoinedSequencesWithQuality]
                         The demultiplexed sequences to be summarized.
                                                                    [required]
Parameters:
  --p-n INTEGER          The number of sequences that should be selected at
                         random for quality score plots. The quality plots
                         will present the average positional qualities across
                         all of the sequences selected. If input sequences are
                         paired end, plots will be generated for both forward
                         and reverse reads for the same `n` sequences.
                                                              [default: 

In [ ]:
# DADA2 truncation sweep

INPUT = f"{QC_DIR}/demux.qza"
THREADS = 2
TRUNCS = [180] # Modify or add truncation values. Feel free to test multiple values!

for T in TRUNCS:
    OUTDIR = f"{PATH_results}/dada2_trunc_{T}"

    print(f"Running denoise-single with trunc-len={T}")

    os.makedirs(OUTDIR, exist_ok=True)

    qiime(
    f"qiime dada2 denoise-single "
    f"--i-demultiplexed-seqs {INPUT} "
    f"--p-trunc-len {T} "
    f"--p-n-threads {THREADS} "
    f"--o-table {OUTDIR}/table.qza "
    f"--o-representative-sequences {OUTDIR}/rep-seqs.qza "
    f"--o-denoising-stats {OUTDIR}/stats.qza "
    f"--verbose"
    )

print("All DADA2 runs completed!")

# Select truncation

SELECTED_T = 180
DADA2_RESULT_DIR = f"{PATH_results}/dada2_trunc_{SELECTED_T}"

print(f"Using results from truncation length: {SELECTED_T}")

# 4. Visualize denoising stats

qiime(f"""
qiime metadata tabulate \
  --m-input-file {DADA2_RESULT_DIR}/stats.qza \
  --o-visualization {DADA2_RESULT_DIR}/stats.qzv
""")

Running denoise-single with trunc-len=180

Usage: qiime dada2 denoise-single [OPTIONS]

  This method denoises single-end sequences, dereplicates them, and filters
  chimeras.

Inputs:
  --i-demultiplexed-seqs ARTIFACT SampleData[SequencesWithQuality |
    PairedEndSequencesWithQuality]
                          The single-end demultiplexed sequences to be
                          denoised.                                 [required]
Parameters:
  --p-trunc-len INTEGER   Position at which sequences should be truncated due
                          to decrease in quality. This truncates the 3' end of
                          the of the input sequences, which will be the bases
                          that were sequenced in the last cycles. Reads that
                          are shorter than this value will be discarded. If 0
                          is provided, no truncation or length filtering will
                          be performed                              [required]
  -

1

In [ ]:
# Filter samples (low frequency)

qiime(
    f"qiime feature-table filter-samples "
    f"--i-table {DADA2_RESULT_DIR}/table.qza "
    f"--p-min-frequency {2000} "
    f"--o-filtered-table {DADA2_RESULT_DIR}/table-filtered-samples.qza"
)

# ================================
# Filter low-abundance features
# ================================

qiime(
    f"qiime feature-table filter-features "
    f"--i-table {DADA2_RESULT_DIR}/table-filtered-samples.qza "
    f"--p-min-frequency {25} "
    f"--o-filtered-table {DADA2_RESULT_DIR}/table-filtered-final.qza"
)

# ================================
# Summarize
# ================================

qiime(
    f"qiime feature-table summarize "
    f"--i-table {DADA2_RESULT_DIR}/table-filtered-final.qza "
    f"--o-visualization {DADA2_RESULT_DIR}/summary-filtered-final.qzv"
)

print("Filtering and summarization complete!")

Saved FeatureTable[Frequency] to: /content/driveDL/MyDrive/qiime2_in_colab/results/dada2_180/table-filtered-samples.qza

Saved FeatureTable[Frequency] to: /content/driveDL/MyDrive/qiime2_in_colab/results/dada2_180/table-filtered-final.qza

Saved Visualization to: /content/driveDL/MyDrive/qiime2_in_colab/results/dada2_180/summary-filtered-final.qzv

Filtering and summarization complete!


## Export for PICRUSt2

In [ ]:
# Define export directory
EXPORT_TO_PICRUST_DIR = f"{DADA2_RESULT_DIR}/export_to_picrust2"
os.makedirs(EXPORT_TO_PICRUST_DIR, exist_ok=True)

print(f"Exporting files to: {EXPORT_TO_PICRUST_DIR}")

# Export Feature table
qiime(
    f"qiime tools export \
  --input-path {EXPORT_TO_PICRUST_DIR}/table-filtered-final.qza \
  --output-path {EXPORT_TO_PICRUST_DIR}/feature-table"
)

# Export Representative sequences
qiime(
    f"qiime tools export \
  --input-path {EXPORT_TO_PICRUST_DIR}/rep-seqs.qza \
  --output-path {EXPORT_TO_PICRUST_DIR}/rep-seqs"
)

print("Export for PICRUSt2 complete.")

Exporting files to: /content/driveDL/MyDrive/qiime2_in_colab/results/picrust2_export
Exported /content/driveDL/MyDrive/qiime2_in_colab/results/dada2_180/table-filtered-final.qza as BIOMV210DirFmt to directory /content/driveDL/MyDrive/qiime2_in_colab/results/picrust2_export/feature-table

Exported /content/driveDL/MyDrive/qiime2_in_colab/results/dada2_180/rep-seqs.qza as DNASequencesDirectoryFormat to directory /content/driveDL/MyDrive/qiime2_in_colab/results/picrust2_export/rep-seqs

Export for PICRUSt2 complete.


# Optional:Save QIIME2 results in YOUR Google Drive

## Mount YOUR Google Drive

In [ ]:
# Optional Drive export

from google.colab import drive
from pathlib import Path
import os
import shutil


SAVE_TO_DRIVE = True

# Drive folder name
drive_folder = "qiime2_in_colab"

def mount_drive():

    drive.mount('/content/driveDL', force_remount=True)

    project_root_drive = Path(
        f"/content/driveDL/MyDrive/{drive_folder}"
    )

    project_root_drive.mkdir(
        parents=True,
        exist_ok=True
    )

    return project_root_drive

## Save QIIME2 results in YOUR Google Drive

In [ ]:
# Save exported PICRUSt2 files to Drive
if SAVE_TO_DRIVE:

    project_root_drive = mount_drive()

    drive_export_path = (
        project_root_drive / "results/picrust2_export"
    )

    shutil.copytree(
        PATH_export_to_picrust2,
        drive_export_path,
        dirs_exist_ok=True
    )

    print(f"Files copied to:\n{drive_export_path}")